In [1]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np

# Load data
data = fetch_california_housing()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

In [2]:
from sklearn.model_selection import train_test_split
import xgboost as xgb

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'seed': 42
}
model = xgb.train(params, dtrain, num_boost_round=100)

dval = xgb.DMatrix(X_test, label=y_test)
model = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=[(dval, 'validation')],
    early_stopping_rounds=10,
    verbose_eval=50
)

xgb.plot_importance(model)
xgb.plot_tree(model, num_trees=0)

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

preds = model.predict(dtest)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"RMSE: {rmse:.4f}, R²: {r2:.4f}")